# 📦 Partie 4 : Création du Data Lake (AWS S3)

## 🎯 Objectif
Sécuriser les données brutes collectées (CSV) en les envoyant vers le Cloud AWS. Cette étape garantit la pérennité des données avant leur transformation.

## 🛠 Choix Techniques
* **AWS S3** : Standard industriel pour le stockage d'objets (Data Lake), fiable et peu coûteux.
* **Boto3** : Utilisation du SDK AWS pour Python pour automatiser l'envoi.
* **Sécurité** : Les identifiants AWS (`Access Key` / `Secret Key`) sont gérés via des variables d'environnement (`.env`) pour ne jamais être exposés dans le code source.

In [1]:
import boto3
import pandas as pd
import os
from dotenv import load_dotenv

# 1. Chargement des secrets
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")

if not AWS_ACCESS_KEY_ID or not BUCKET_NAME:
    raise ValueError("❌ Erreur : Identifiants AWS manquants dans le .env")

# 2. Connexion à S3
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="eu-west-3" # Paris
)
s3 = session.client("s3")

# 3. Création du CSV enrichi unique (météo + hôtels fusionnés par ville)
# L'énoncé demande "a .csv file containing enriched information about weather and hotels"
print("🔗 Fusion des données météo + hôtels en un fichier enrichi...")
df_weather = pd.read_csv('cities_weather_data_7days.csv')
df_hotels = pd.read_csv('hotels_data.csv')

df_enriched = df_hotels.merge(
    df_weather, left_on='city', right_on='city_name', how='left', suffixes=('', '_weather')
)
# Nettoyage : supprimer la colonne city_name dupliquée
df_enriched = df_enriched.drop(columns=['city_name'], errors='ignore')
df_enriched.to_csv('kayak_enriched_data.csv', index=False)
print(f"   ✅ kayak_enriched_data.csv créé ({len(df_enriched)} lignes)")

# 4. Upload de tous les fichiers vers S3
files_to_upload = [
    "cities_coordinates.csv",
    "cities_weather_data_7days.csv",
    "hotels_data.csv",
    "kayak_enriched_data.csv"       # Fichier enrichi unique (demandé par l'énoncé)
]

print(f"\n📦 Début du transfert vers le bucket '{BUCKET_NAME}'...")

for file_name in files_to_upload:
    try:
        if os.path.exists(file_name):
            print(f"   ⬆️ Envoi de {file_name}...")
            s3.upload_file(file_name, BUCKET_NAME, file_name)
            print(f"   ✅ {file_name} envoyé avec succès !")
        else:
            print(f"   ⚠️ Fichier local introuvable : {file_name}")
            
    except Exception as e:
        print(f"   ❌ Erreur d'upload sur {file_name} : {e}")

print("\n🎉 Transfert vers le Data Lake terminé.")

🔗 Fusion des données météo + hôtels en un fichier enrichi...
   ✅ kayak_enriched_data.csv créé (700 lignes)

📦 Début du transfert vers le bucket 'kayak-project-aws'...
   ⬆️ Envoi de cities_coordinates.csv...
   ✅ cities_coordinates.csv envoyé avec succès !
   ⬆️ Envoi de cities_weather_data_7days.csv...
   ✅ cities_weather_data_7days.csv envoyé avec succès !
   ⬆️ Envoi de hotels_data.csv...
   ✅ hotels_data.csv envoyé avec succès !
   ⬆️ Envoi de kayak_enriched_data.csv...
   ✅ kayak_enriched_data.csv envoyé avec succès !

🎉 Transfert vers le Data Lake terminé.
